# Working on measuring the ecological variaitons of 24 NEON sites

In [1]:
from google.auth import compute_engine
import ee
print(ee.__version__)
import os

import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import folium

0.1.370


In [2]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='rse-global-wetlands') # this is the esential step!

Enter verification code:  4/1AfrIepBOmHydB5cjyafNXwg9cXYUdncsmJ74nEomxvHUV8rYEtopCRVXumo



Successfully saved authorization token.


In [7]:
import ee

# ---------------------------------------
# 1 Initialize Earth Engine
# ---------------------------------------
ee.Initialize()

# -----------------------------------------------------
# 2 Load NEON Tree / Plot Polygons
# -----------------------------------------------------
trees = ee.FeatureCollection(
    'projects/rse-global-wetlands/assets/All_NEON_TOS_Plot_Polygons_V11'
)

# Optional: filter to SOAP site if attribute exists
# trees = trees.filter(ee.Filter.eq('siteID', 'SOAP'))

# -----------------------------------------------------
# 3 Read NEON CHM Image Collection
# -----------------------------------------------------
chm = ee.ImageCollection(
    'projects/neon-prod-earthengine/assets/CHM/001'
)

# Print available image IDs
print('NEON CHM Images:')
print(chm.aggregate_array('system:index').getInfo())

# -----------------------------------------------------
# 4 Filter to July 2020 and SOAP site
# -----------------------------------------------------
# These is no image in 2020-7
start_date = ee.Date('2021-07-01')
end_date = ee.Date('2021-08-01')

chm_july2020 = (
    chm
    .filterDate(start_date, end_date)
    .filter(ee.Filter.eq('NEON_SITE', 'SOAP'))
)

# -----------------------------------------------------
# 5 Create mean CHM composite
# -----------------------------------------------------
chm_image = chm_july2020.mean()

# -----------------------------------------------------
# 6 Extract mean CHM per polygon
# -----------------------------------------------------
chm_stats = chm_image.reduceRegions(
    collection=trees,
    reducer=ee.Reducer.mean(),
    scale=1,
    tileScale=4
)

# Print preview
print("Sample output:")
print(chm_stats.limit(5).getInfo())

# -----------------------------------------------------
# 7 Export results to Google Drive
# -----------------------------------------------------
task = ee.batch.Export.table.toDrive(
    collection=chm_stats,
    description='SOAP_Tree_CHM_July2021',
    fileFormat='CSV'
)

task.start()

print("Export task started.")

NEON CHM Images:
['2013_CPER_1', '2013_SJER_1', '2013_SOAP_1', '2013_STER_1', '2013_TEAK_1', '2014_BART_1', '2014_DSNY_1', '2014_HARV_2', '2014_OSBS_1', '2015_DELA_1', '2015_GRSM_1', '2015_LENO_1', '2015_MLBS_1', '2015_ORNL_1', '2015_TALL_1', '2016_BART_2', '2016_BLAN_1', '2016_CHEQ_1', '2016_CLBJ_1', '2016_DELA_2', '2016_DSNY_2', '2016_GRSM_2', '2016_HARV_3', '2016_HOPB_1', '2016_JERC_2', '2016_KONZ_1', '2016_LENO_2', '2016_NOGP_1', '2016_OAES_1', '2016_ORNL_2', '2016_OSBS_2', '2016_PRIN_2', '2016_SCBI_1', '2016_SERC_1', '2016_STEI_1', '2016_TALL_2', '2016_UKFS_1', '2016_UNDE_1', '2016_WOOD_1', '2017_ABBY_1', '2017_ARIK_1', '2017_BARR_1', '2017_BART_3', '2017_BLAN_2', '2017_BLUE_1', '2017_BONA_1', '2017_CHEQ_2', '2017_CLBJ_2', '2017_CPER_3', '2017_DEJU_1', '2017_DELA_3', '2017_DSNY_3', '2017_GRSM_3', '2017_HARV_4', '2017_HEAL_1', '2017_HOPB_2', '2017_JERC_3', '2017_JORN_1', '2017_KONZ_2', '2017_LENO_3', '2017_LIRO_1', '2017_MCDI_1', '2017_MLBS_2', '2017_MOAB_1', '2017_NIWO_1', '2017_N

In [5]:
import ee
ee.Initialize()

chm = ee.ImageCollection('projects/neon-prod-earthengine/assets/CHM/001')

# filter site first
soap_chm = chm.filter(ee.Filter.eq('NEON_SITE', 'SOAP'))

print("Available images for SOAP:")
print(soap_chm.aggregate_array('system:index').getInfo())

Available images for SOAP:
['2013_SOAP_1', '2017_SOAP_2', '2018_SOAP_3', '2019_SOAP_4', '2021_SOAP_5', '2023_SOAP_6', '2023_SOAP_7', '2024_SOAP_8']


In [14]:
import ee
import geemap

# ---------------------------------------------
# 1 Initialize Earth Engine
# ---------------------------------------------
ee.Initialize()

# ---------------------------------------------
# 2 Load NEON Hyperspectral Reflectance
# ---------------------------------------------
refl002 = ee.ImageCollection(
    "projects/neon-prod-earthengine/assets/HSI_REFL/002"
)

print("NEON Hyperspectral Images:")
print(refl002.aggregate_array('system:index').getInfo())

# ---------------------------------------------
# 3 Filter July 2020
# ---------------------------------------------
start_date = ee.Date('2020-07-01')
end_date = ee.Date('2020-08-01')

refl2020 = refl002.filterDate(start_date, end_date)

# Filter by site
reflLIRO = refl2020.filter(ee.Filter.eq('NEON_SITE', 'LIRO'))

# ---------------------------------------------
# 4 Convert ImageCollection → Image
# ---------------------------------------------
refl_image = reflLIRO.mean()

# print("Band names:")
# print(refl_image.bandNames().getInfo())

# ---------------------------------------------
# 5 Vegetation Indices
# ---------------------------------------------

# NDVI
NDVI = refl_image.normalizedDifference(['B095','B053']).rename('NDVI')

# EVI
EVI = refl_image.expression(
    '2.5 * ((nir - red) / (nir + 6 * red - 7.5 * blue + 1))',
    {
        'nir': refl_image.select('B095'),
        'red': refl_image.select('B053'),
        'blue': refl_image.select('B019')
    }
).rename('EVI')

# NDWI
NDWI = refl_image.normalizedDifference(['B095','B140']).rename('NDWI')

# PRI
PRI = refl_image.normalizedDifference(['B031','B029']).rename('PRI')

# ---------------------------------------------
# 6 Spectral Diversity Metric
# ---------------------------------------------
spectral_var = refl_image.reduceNeighborhood(
    reducer=ee.Reducer.variance(),
    kernel=ee.Kernel.square(3)
).rename('SpectralVariance')

# ---------------------------------------------
# 7 Stack indices
# ---------------------------------------------
indices = NDVI.addBands([
    EVI,
    NDWI,
    PRI,
    spectral_var
])

print("Indices image:")
# print(indices)

# ---------------------------------------------
# 8 Visualization (optional)
# ---------------------------------------------
Map = geemap.Map()

rgb_vis = {
    'min':340,
    'max':2150,
    'bands':['B053','B035','B019'],
    'gamma':2
}

Map.addLayer(refl_image, rgb_vis, 'LIRO July 2020 Reflectance')

Map.addLayer(
    NDVI,
    {'min':-0.2,'max':0.9,'palette':['blue','white','green']},
    'NDVI'
)

Map.addLayer(
    EVI,
    {'min':0,'max':1,'palette':['white','yellow','darkgreen']},
    'EVI'
)

Map.centerObject(refl_image, 14)

Map

# ---------------------------------------------
# 9 Extract plot statistics
# ---------------------------------------------
plots = ee.FeatureCollection(
    'projects/rse-global-wetlands/assets/All_NEON_TOS_Plot_Polygons_V11'
)

# Optional site filter
# plots = plots.filter(ee.Filter.eq('siteID','LIRO'))

stats = indices.reduceRegions(
    collection=plots,
    reducer=ee.Reducer.mean(),
    scale=1,
    tileScale=4
)

print("Sample output:")
# print(stats.limit(5).getInfo())

# ---------------------------------------------
# 10 Export results
# ---------------------------------------------
task = ee.batch.Export.table.toDrive(
    collection=stats,
    description='HSI_REFL_2020_July_VegIndices',
    fileFormat='CSV'
)

task.start()

print("Export started.")

NEON Hyperspectral Images:
['2022_ABBY_5', '2022_ARIK_4', '2022_BART_6', '2022_BLAN_5', '2022_CHEQ_5', '2022_CLBJ_6', '2022_CUPE_2', '2022_GRSM_7', '2022_GUAN_2', '2022_GUIL_2', '2022_HARV_7', '2022_HOPB_4', '2022_JORN_5', '2022_KONZ_7', '2022_LAJA_2', '2022_LIRO_3', '2022_MCDI_3', '2022_MCRA_3', '2022_MLBS_5', '2022_MOAB_6', '2022_OAES_6', '2022_ONAQ_4', '2022_PRIN_5', '2022_REDB_4', '2022_RMNP_4', '2022_SCBI_5', '2022_SERC_6', '2022_SRER_5', '2022_STEI_5', '2022_STER_4', '2022_TOOL_4', '2022_UKFS_6', '2022_UNDE_5', '2022_WREF_5', '2022_YELL_4', '2023_ABBY_6', '2023_ARIK_5', '2023_BONA_5', '2023_CLBJ_7', '2023_DEJU_5', '2023_DELA_7', '2023_DSNY_7', '2023_HEAL_5', '2023_JERC_7', '2023_KONZ_8', '2023_LENO_7', '2023_MCRA_4', '2023_MLBS_6', '2023_MOAB_7', '2023_NIWO_5', '2023_NOGP_6', '2023_OAES_7', '2023_ONAQ_5', '2023_OSBS_7', '2023_PRIN_6', '2023_REDB_5', '2023_RMNP_5', '2023_SCBI_6', '2023_SJER_6', '2023_SOAP_6', '2023_SOAP_7', '2023_TALL_7', '2023_TEAK_6', '2023_UKFS_7', '2023_WLOU_4

EEException: Image.normalizedDifference: No band named 'B095'. Available band names: [].

In [15]:
import ee
ee.Initialize()

refl002 = ee.ImageCollection(
    "projects/neon-prod-earthengine/assets/HSI_REFL/002"
)

print("All available hyperspectral images:")
print(refl002.aggregate_array('system:index').getInfo())

All available hyperspectral images:
['2022_ABBY_5', '2022_ARIK_4', '2022_BART_6', '2022_BLAN_5', '2022_CHEQ_5', '2022_CLBJ_6', '2022_CUPE_2', '2022_GRSM_7', '2022_GUAN_2', '2022_GUIL_2', '2022_HARV_7', '2022_HOPB_4', '2022_JORN_5', '2022_KONZ_7', '2022_LAJA_2', '2022_LIRO_3', '2022_MCDI_3', '2022_MCRA_3', '2022_MLBS_5', '2022_MOAB_6', '2022_OAES_6', '2022_ONAQ_4', '2022_PRIN_5', '2022_REDB_4', '2022_RMNP_4', '2022_SCBI_5', '2022_SERC_6', '2022_SRER_5', '2022_STEI_5', '2022_STER_4', '2022_TOOL_4', '2022_UKFS_6', '2022_UNDE_5', '2022_WREF_5', '2022_YELL_4', '2023_ABBY_6', '2023_ARIK_5', '2023_BONA_5', '2023_CLBJ_7', '2023_DEJU_5', '2023_DELA_7', '2023_DSNY_7', '2023_HEAL_5', '2023_JERC_7', '2023_KONZ_8', '2023_LENO_7', '2023_MCRA_4', '2023_MLBS_6', '2023_MOAB_7', '2023_NIWO_5', '2023_NOGP_6', '2023_OAES_7', '2023_ONAQ_5', '2023_OSBS_7', '2023_PRIN_6', '2023_REDB_5', '2023_RMNP_5', '2023_SCBI_6', '2023_SJER_6', '2023_SOAP_6', '2023_SOAP_7', '2023_TALL_7', '2023_TEAK_6', '2023_UKFS_7', '20

In [9]:
import ee
ee.Initialize()

refl002 = ee.ImageCollection(
    "projects/neon-prod-earthengine/assets/HSI_REFL/002"
)

# Check available LIRO flights
liro_images = refl002.filter(ee.Filter.eq('NEON_SITE', 'LIRO'))

print(liro_images.aggregate_array('system:index').getInfo())  #this is no image in this period

['2022_LIRO_3', '2024_LIRO_4', '2025_LIRO_5']
